# Gradient/flux module vs analytic ground truth

The `tl._gradient` module's statistics are either novel (`interface_flux`)
or assembled from published pieces in a way no external package reproduces
(`expression_gradient`'s graph-MLS, `decay_length`'s offset exponential fit,
`gradient_significance`'s autocorrelation-matched surrogate null). There is
therefore no reference *package* to benchmark against — **synthetic fields
with closed-form answers are the reference**. Every section plants a field
whose gradient, decay length, divergence, interface flux, or null behavior
is known exactly, and checks the recovered value.

**Validated here:**

1. `expression_gradient` — exact recovery of a planted linear gradient
   (a linear field lies in the MLS model space, so recovery must be exact
   to float precision), plus noise behavior: unbiased ensemble mean, and the
   documented magnitude-shrinkage cost of diffusion pre-smoothing.
2. `decay_length` — recovery of λ, the background offset c, and a sane
   standard error from a planted radial exponential.
3. `gradient_field` — negative divergence at a planted expression peak
   (∇²f < 0 at a maximum, matching the documented sign convention), and the
   absence of any `curl` output (curl-free by construction).
4. `interface_flux` — three closed-form targets on a planted linear ramp
   crossing a two-block label boundary, including a Poisson–Voronoi
   geometry identity, plus bit-exact antisymmetry.
5. `gradient_significance` — Type-I calibration under a spatially
   autocorrelated but directionally isotropic null (the case the old CSR
   null failed), and a directional positive control at the permutation floor.

**Not validated here:** real-tissue behavior (no ground truth exists there);
regimes that violate each estimator's assumptions (e.g. expression *steps*
at a boundary, where a local linear fit is documented to be weakest);
`spatial_diffusion`'s Sepal score, which has its own group
(`benchmarks/sepal/` — a self-referential regression test, see the README).
The usual astrolabe-vs-reference timing comparison does not apply: there is
no reference implementation to time.

In [1]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import anndata as ad
import astrolabe as ab


def make_adata(coords, genes: dict, n_neighbors=8):
    """AnnData with planted expression columns + a spatial KNN graph."""
    X = np.column_stack(list(genes.values())).astype(np.float32)
    adata = ad.AnnData(X=X)
    adata.var_names = list(genes.keys())
    adata.obsm["spatial"] = coords.astype(np.float32)
    ab.tl.spatial_neighbors(adata, n_neighbors=n_neighbors)
    return adata

rng = np.random.default_rng(0)

INFO:astrolabe.pl._theme:Registered bundled Inter font files with matplotlib


INFO:astrolabe.pl._theme:astrolabe plot theme set to 'light'


## 1. Linear ramp: f = a·x + b·y  →  ∇f = (a, b) everywhere

3000 cells uniform in [0, 500]² µm, a = 0.002, b = −0.001 (log1p-scale
values per µm — realistic magnitudes). Because a linear field is *inside*
the MLS model space, the noiseless recovery must be exact to float
precision — any systematic error would be an implementation bug, not
estimator variance. A noisy replicate (σ = 0.02 additive Gaussian) then
separates variance from bias: per-cell estimates scatter, but the ensemble
mean must stay on target.

In [2]:
n = 3000
coords = rng.uniform(0, 500, (n, 2))
a_true, b_true = 0.002, -0.001
g_true = np.array([a_true, b_true])
f_ramp = a_true * coords[:, 0] + b_true * coords[:, 1]
f_noisy = f_ramp + rng.normal(0, 0.02, n)

adata = make_adata(coords, {"ramp": f_ramp, "ramp_noisy": f_noisy})
ab.tl.expression_gradient(adata, ["ramp", "ramp_noisy"], n_diffusion_steps=0)
reliable = ~adata.obs["gradient_unreliable"].to_numpy()

g = adata.obsm["gradient_ramp"][reliable]
err = np.linalg.norm(g - g_true, axis=1) / np.linalg.norm(g_true)
print(f"noiseless ramp: median relative vector error = {np.median(err):.2e} "
      f"(n reliable = {reliable.sum()}/{n})")
assert np.median(err) < 1e-4, "linear field must be recovered exactly"

gn = adata.obsm["gradient_ramp_noisy"][reliable]
err_n = np.linalg.norm(gn - g_true, axis=1) / np.linalg.norm(g_true)
mean_g = gn.mean(axis=0)
bias = np.linalg.norm(mean_g - g_true) / np.linalg.norm(g_true)
print(f"noisy ramp (steps=0): median per-cell error = {np.median(err_n):.3f}, "
      f"ensemble mean = ({mean_g[0]:.5f}, {mean_g[1]:.5f}), rel. bias = {bias:.3f}")
assert bias < 0.05, "noisy-ramp ensemble mean should be unbiased"

# The documented cost of diffusion pre-smoothing: less per-cell variance,
# but magnitudes shrink by a graph-dependent factor (docstring: 'magnitudes
# are relative, not physical').
ab.tl.expression_gradient(adata, "ramp_noisy", n_diffusion_steps=1)
gs = adata.obsm["gradient_ramp_noisy"][reliable]
err_s = np.linalg.norm(gs - g_true, axis=1) / np.linalg.norm(g_true)
shrink = np.linalg.norm(gs.mean(axis=0)) / np.linalg.norm(g_true)
print(f"noisy ramp (steps=1): median per-cell error = {np.median(err_s):.3f}, "
      f"mean-magnitude shrinkage factor = {shrink:.2f}")

INFO:astrolabe.tl._graph:spatial_neighbors: 3000 cells, 27904 edges (mean degree 9.3) [generic]


INFO:astrolabe.tl._gradient:expression_gradient: 481/3000 cells flagged gradient_unreliable (boundary/one-sided neighborhoods)


INFO:astrolabe.tl._gradient:expression_gradient: 'ramp' — mean magnitude 0.0022, directional coherence R̄=1.000


INFO:astrolabe.tl._gradient:expression_gradient: 'ramp_noisy' — mean magnitude 0.0025, directional coherence R̄=0.862


INFO:astrolabe.tl._gradient:expression_gradient: 481/3000 cells flagged gradient_unreliable (boundary/one-sided neighborhoods)


INFO:astrolabe.tl._gradient:expression_gradient: 'ramp_noisy' — mean magnitude 0.0019, directional coherence R̄=0.955


noiseless ramp: median relative vector error = 3.81e-07 (n reliable = 2519/3000)
noisy ramp (steps=0): median per-cell error = 0.511, ensemble mean = (0.00198, -0.00100), rel. bias = 0.011
noisy ramp (steps=1): median per-cell error = 0.289, mean-magnitude shrinkage factor = 0.83


The noiseless gradient is recovered to ~1e-7 relative error — machine noise
from the float32 output dtype, i.e. exact. Under additive noise the
per-cell estimates are individually noisy (median error ~50% at this SNR:
differentiation amplifies noise at the ~30 µm neighborhood scale) but the
ensemble mean is unbiased to <1%, which is the property downstream
aggregators (`interface_flux`, `gradient_significance`) actually rely on.
One diffusion step cuts the per-cell error by nearly half while shrinking
the mean magnitude ~15–20% — exactly the trade-off the docstring warns about
(magnitudes comparable only within one `n_diffusion_steps` setting).

## 2. Radial exponential: f = f₀·e^(−r/λ) + c with λ = 80 µm, c = 0.5

5000 cells uniform in [0, 800]² µm, source at (400, 400), f₀ = 2.0,
additive noise σ = 0.1, fit window 400 µm (5λ — the identifiable regime).
The background offset c is the term whose absence used to drive fitted λ
toward infinity on real data; the planted c = 0.5 checks it is actually
estimated, and `lambda_se_um` checks the fit covariance is propagated.

In [3]:
n = 5000
coords_r = rng.uniform(0, 800, (n, 2))
r = np.linalg.norm(coords_r - [400.0, 400.0], axis=1)
lam_true, f0_true, c_true = 80.0, 2.0, 0.5
expr = f0_true * np.exp(-r / lam_true) + c_true + rng.normal(0, 0.1, n)

adata_r = make_adata(coords_r, {"radial": expr})
ab.tl.decay_length(adata_r, "radial", source=(400, 400), max_dist_um=400)
res = adata_r.uns["decay_length_radial"]
print(f"lambda = {res['lambda_um']:.1f} ± {res['lambda_se_um']:.1f} µm (true {lam_true}), "
      f"background = {res['background']:.3f} (true {c_true}), "
      f"f0 = {res['f0']:.3f} (true {f0_true}), R² = {res['r_squared']:.3f}")

assert abs(res["lambda_um"] - lam_true) < 8, "lambda not recovered within 10%"
assert abs(res["background"] - c_true) < 0.1, "background offset not recovered"
assert res["lambda_se_um"] < 0.5 * res["lambda_um"], "lambda SE should be small in-window"

INFO:astrolabe.tl._graph:spatial_neighbors: 5000 cells, 46358 edges (mean degree 9.3) [generic]


INFO:astrolabe.tl._gradient:decay_length: 'radial' λ=79.8 µm, f₀=1.983, R²=0.999


lambda = 79.8 ± 0.9 µm (true 80.0), background = 0.501 (true 0.5), f0 = 1.983 (true 2.0), R² = 0.999


## 3. Gaussian bump: divergence < 0 at the peak, and no `curl` output

f = A·exp(−r²/2σ²) with A = 2, σ = 100 µm, peaked at (250, 250). For a
gradient field, divergence = ∇²f, and analytically
∇²f(peak) = −2A/σ² = −4·10⁻⁴ — **negative at a maximum**, matching the
documented sign convention (streamlines converge into peaks). The field
dict must also contain no `curl` entry: the overhauled `gradient_field`
interpolates the *scalar* field and differentiates it, so its output is
curl-free by construction and a curl map would only ever have measured the
interpolator.

In [4]:
n = 3000
coords_b = rng.uniform(0, 500, (n, 2))
A_true, sig = 2.0, 100.0
f_bump = A_true * np.exp(-((coords_b - 250.0) ** 2).sum(axis=1) / (2 * sig**2))

adata_b = make_adata(coords_b, {"bump": f_bump})
ab.tl.expression_gradient(adata_b, "bump", n_diffusion_steps=0)
ab.tl.gradient_field(adata_b, "bump", n_grid=40)

fld = adata_b.uns["gradient_field_bump"]
ix = np.argmin(np.abs(fld["X_grid"] - 250.0))
iy = np.argmin(np.abs(fld["Y_grid"] - 250.0))
div_peak = fld["divergence"][iy, ix]
div_analytic = -2 * A_true / sig**2
print(f"divergence at peak = {div_peak:.2e} (analytic ∇²f = {div_analytic:.2e})")
print(f"'curl' in field dict: {'curl' in fld}")

assert div_peak < 0, "divergence must be negative at an expression maximum"
assert abs(div_peak - div_analytic) < 0.25 * abs(div_analytic), \
    "peak divergence should be within 25% of the analytic Laplacian"
assert "curl" not in fld, "gradient_field must not emit a curl map"

INFO:astrolabe.tl._graph:spatial_neighbors: 3000 cells, 27792 edges (mean degree 9.3) [generic]


INFO:astrolabe.tl._gradient:expression_gradient: 433/3000 cells flagged gradient_unreliable (boundary/one-sided neighborhoods)


INFO:astrolabe.tl._gradient:expression_gradient: 'bump' — mean magnitude 0.0058, directional coherence R̄=0.013


INFO:astrolabe.tl._gradient:gradient_field: gridding scalar field to 40 × 40 …


INFO:astrolabe.tl._gradient:gradient_field: 'bump' — grid 40×40, tissue coverage 98.0%


divergence at peak = -3.47e-04 (analytic ∇²f = -4.00e-04)
'curl' in field dict: False


## 4. Two-block interface: three closed-form targets for `interface_flux`

4000 cells uniform in [0, 500]² µm, labels A (x < 250) and B (x ≥ 250), and
a linear ramp f = a·x with a = 0.004 crossing the boundary (the field is
smooth — the *labels* step, not the expression, so the MLS gradient is
exact everywhere and the flux targets are closed-form):

1. **Integrated flux** Φ(A→B) = ∮ (∂f/∂n) dS. Each facet contributes
   L·a·n̂ₓ = a × (its projection onto the interface direction), and the
   signed projections telescope along the facet chain, so
   Φ(A→B) ≈ a × H = 0.004 × 500 = **2.0** — independent of how jagged the
   chain is.
2. **Interface length**: the Voronoi facet chain separating two half-plane
   populations of uniform-random points is jagged, so its length must
   *exceed* the planted 500 µm — by the factor **4/π ≈ 1.273** expected for
   paths in a Poisson–Voronoi tessellation (Baccelli, Tchoumatchenko &
   Zuyev 2000), giving ≈ 637 µm. (Consequently the length-weighted mean
   normal derivative is ≈ a·π/4, not a — the projection factor moves from
   the numerator to the denominator.)
3. **Antisymmetry**: Φ(B→A) = −Φ(A→B) must hold **bit-exactly** — the
   docstring states it as an identity (same facets, flipped normal).

One benchmark-specific choice: `exclude_unreliable=False`. The boundary flag (one-sided neighborhoods) correctly marks the rim band of a uniform square, and the interface chain *ends* on that rim — with the default exclusion the chain is truncated and the telescoped flux undershoots a·H. Because the planted field is exactly affine, the free-intercept fit is exact even at flagged cells, so including them is valid here; on real (noisy) data the default exclusion is the right call.

In [5]:
n = 4000
coords_f = rng.uniform(0, 500, (n, 2))
a_step = 0.004
f_lin = a_step * coords_f[:, 0]

adata_f = make_adata(coords_f, {"grad_gene": f_lin})
adata_f.obs["block"] = np.where(coords_f[:, 0] < 250, "A", "B")
ab.tl.expression_gradient(adata_f, "grad_gene", n_diffusion_steps=0)
# exclude_unreliable=False: the closed form integrates over the FULL
# 500 µm interface, but the boundary flag correctly marks the rim band
# of this uniform square (one-sided neighborhoods), truncating the
# chain's ends. The planted field is exactly affine, so even flagged
# cells' free-intercept fits are exact — including them is valid here
# and required for the full-chain target. On real data keep the default.
ab.tl.interface_flux(adata_f, "block", "grad_gene", exclude_unreliable=False)

fx = adata_f.uns["interface_flux"]
flux = fx["integrated_flux"]["grad_gene"]
mnd = fx["mean_normal_derivative"]["grad_gene"]
L = fx["interface_length"]

phi_ab = flux.loc["A", "B"]
L_ab = L.loc["A", "B"]
print(f"integrated flux Φ(A→B) = {phi_ab:.3f}   (closed form a·H = {a_step * 500:.3f})")
print(f"interface length = {L_ab:.1f} µm         (4/π × 500 = {4 / np.pi * 500:.1f})")
print(f"mean normal derivative = {mnd.loc['A', 'B']:.5f} (a·π/4 = {a_step * np.pi / 4:.5f})")
print(f"Φ(B→A) = {flux.loc['B', 'A']:.3f}; bit-exact antisymmetry: "
      f"{flux.loc['B', 'A'] == -phi_ab}")

assert abs(phi_ab - a_step * 500) < 0.1 * (a_step * 500), "flux off closed form by >10%"
assert 1.15 < L_ab / 500 < 1.40, "interface length inconsistent with 4/π excess"
assert flux.loc["B", "A"] == -phi_ab, "antisymmetry identity must be bit-exact"
assert L.loc["B", "A"] == L_ab, "interface length must be bit-exact symmetric"

INFO:astrolabe.tl._graph:spatial_neighbors: 4000 cells, 37014 edges (mean degree 9.3) [generic]


INFO:astrolabe.tl._gradient:expression_gradient: 549/4000 cells flagged gradient_unreliable (boundary/one-sided neighborhoods)


INFO:astrolabe.tl._gradient:expression_gradient: 'grad_gene' — mean magnitude 0.0040, directional coherence R̄=1.000


INFO:astrolabe.tl._gradient:interface_flux: 130 interface facets (633 µm total) across 2 groups for 1 features


integrated flux Φ(A→B) = 2.074   (closed form a·H = 2.000)
interface length = 633.4 µm         (4/π × 500 = 636.6)
mean normal derivative = 0.00328 (a·π/4 = 0.00314)
Φ(B→A) = -2.074; bit-exact antisymmetry: True


All three targets land: Φ within a few % of a·H (residual excess comes from
the facet chain occasionally back-tracking in y, double-covering short
stretches), interface length within ~1% of the 4/π Poisson–Voronoi
prediction, and antisymmetry exact to the bit. Honest scope note: this
plants a *smooth* field across the label boundary. When the expression
itself steps at the interface — the biologically common case — the local
linear MLS fit smooths the step over the neighborhood scale, and the
docstring's own caveat applies: treat magnitudes as relative comparisons
between pairs, not physical fluxes.

## 5. `gradient_significance` calibration under `null='smooth'`

The overhauled null exists to fix a specific failure: a full label shuffle
(CSR) rejects for *any* spatially autocorrelated field, coherent or not. So
the null case here is exactly that trap — **smoothed isotropic noise**:
white noise diffused 8 steps on the spatial graph, which is strongly
spatially autocorrelated but has no preferred tissue-wide direction. A
calibrated test must *not* call it coherent. 20 independent replicates
(fresh noise each time) give the empirical false-positive rate at
α = 0.05; with `n_perms=99`, the smallest attainable p is 1/100 = 0.01.

The positive control is a directional ramp plus noise — the one pattern R̄
is designed to detect — which must hit the permutation floor.

In [6]:
import scipy.sparse as sp
logging.disable(logging.INFO)  # silence per-surrogate progress for the loop

n = 1500
coords_c = rng.uniform(0, 500, (n, 2))
adata_c = make_adata(coords_c, {"placeholder": np.zeros(n)})
conn = adata_c.obsp["spatial_neighbors_connectivities"]
row_sums = np.asarray(conn.sum(axis=1)).ravel()
row_sums[row_sums == 0] = 1.0
M = sp.diags(1.0 / row_sums) @ conn.astype(np.float64)

n_reps, n_perms = 20, 99
pvals = []
for rep in range(n_reps):
    field = rng.standard_normal(n)
    for _ in range(8):  # autocorrelated but directionally isotropic
        field = M @ field
    adata_c.obs["nullfield"] = field
    ab.tl.expression_gradient(adata_c, "nullfield", n_diffusion_steps=1)
    ab.tl.gradient_significance(adata_c, "nullfield", null="smooth",
                                n_perms=n_perms, random_state=rep)
    pvals.append(float(adata_c.uns["gradient_significance"]
                       .loc["nullfield", "p_coherence"]))
logging.disable(logging.NOTSET)

pvals = np.array(pvals)
frac_sig = float((pvals < 0.05).mean())
print(f"null p_coherence over {n_reps} replicates: {np.round(np.sort(pvals), 3)}")
print(f"fraction < 0.05 = {frac_sig:.2f} (nominal 0.05; "
      f"binomial 95% range for 20 reps: 0–3 hits)")
assert frac_sig <= 0.15, "smooth null is anticonservative"

# positive control: directional ramp + noise
adata_c.obs["rampfield"] = 0.004 * coords_c[:, 0] + rng.normal(0, 0.1, n)
ab.tl.expression_gradient(adata_c, "rampfield", n_diffusion_steps=1)
ab.tl.gradient_significance(adata_c, "rampfield", null="smooth",
                            n_perms=n_perms, random_state=0)
row = adata_c.uns["gradient_significance"].loc["rampfield"]
floor = 1.0 / (n_perms + 1)
print(f"positive control: p_coherence = {row['p_coherence']:.4f} "
      f"(permutation floor {floor:.4f}), "
      f"R̄ = {row['coherence']:.3f} vs null mean {row['coherence_null_mean']:.3f}")
assert row["p_coherence"] == floor, "directional ramp must hit the permutation floor"
assert row["coherence"] > 5 * row["coherence_null_mean"], \
    "effect size should dwarf the null baseline"

INFO:astrolabe.tl._gradient:expression_gradient: 262/1500 cells flagged gradient_unreliable (boundary/one-sided neighborhoods)


INFO:astrolabe.tl._gradient:expression_gradient: 'rampfield' — mean magnitude 0.0035, directional coherence R̄=0.878


INFO:astrolabe.tl._gradient:gradient_significance: 'rampfield' surrogate smoothing calibrated to 12 diffusion steps (graph Moran's I matched)


INFO:astrolabe.tl._gradient:gradient_significance: 'rampfield' running 99 surrogates (n_jobs=1)


INFO:astrolabe.tl._gradient:gradient_significance: 'rampfield' magnitude=0.0036 (p=0.020, two-sided) coherence=0.901 vs null mean 0.103 (p=0.010)


null p_coherence over 20 replicates: [0.18 0.28 0.37 0.48 0.5  0.51 0.55 0.59 0.62 0.63 0.67 0.68 0.85 0.85
 0.86 0.89 0.89 0.93 0.95 0.95]
fraction < 0.05 = 0.00 (nominal 0.05; binomial 95% range for 20 reps: 0–3 hits)
positive control: p_coherence = 0.0100 (permutation floor 0.0100), R̄ = 0.901 vs null mean 0.103


### Interpretation (calibration)

No false positives in 20 null replicates, with p-values spread broadly over
(0, 1] — consistent with the nominal rate (0/20 has probability ≈ 0.36 when
the true rate is exactly 0.05, so 20 replicates only *bound* the rate
coarsely; they cannot distinguish 0.05 from, say, 0.01). If anything the
spread leans conservative, the expected direction for a surrogate null that
matches — or slightly overshoots — the observed field's autocorrelation.
The essential pair of properties holds: a spatially autocorrelated but
directionless field is **not** called coherent (this is precisely the case
where the old CSR null rejected ~always), while a genuinely directional
field is, at the smallest attainable p and with an R̄ effect size ~8× the
null baseline — which is why `coherence_null_mean` is reported alongside p.